In [5]:
# import required packages

import pandas as pd
import os


In [ ]:
# functions for importing and processing the data

def process_cpi_data(file_names: dict[str, str]) -> pd.DataFrame:
    """
    Import and clean the raw cpi data from the csv files and return them in a combined dataframe.
    """

    # prepare the dictionaries to store the dataframes and convert the months to numerical values
    data = []
    months_mapping = {
                'januari': '01', 'februari': '02', 'maart': '03', 'april': '04',
                'mei': '05', 'juni': '06', 'juli': '07', 'augustus': '08',
                'september': '09', 'oktober': '10', 'november': '11', 'december': '12'
            }

    # loop through the file names and import the data
    for file_key, file_name in file_names.items():
        file_path = os.path.join("..", "data", "raw_data", file_name)
        df = pd.read_csv(file_path, delimiter=';', encoding='utf-8', skiprows=4)
        df = df.iloc[:-1, :]    # delete last row

        # convert dates to datetime
        date_clean = df.iloc[:, 0].replace(months_mapping, regex=True)
        df['Date'] = pd.to_datetime(date_clean, format='%Y %m')

        # convert data to floats
        df[f'CPI_{file_key}'] = (
            df.iloc[:, 1]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
            .astype(float)
        )

        data.append(df[['Date', f'CPI_{file_key}']].set_index('Date'))

    combined_df = pd.concat(data, axis=1).reset_index()
    return combined_df

def process_cpi_yearly(cpi_df: pd.DataFrame) -> pd.DataFrame:
    """Yearly means of the monthly CPI data."""
    cpi_year = cpi_df.groupby(cpi_df.index.year).mean()
    cpi_year.index.name = "year"
    return cpi_year

def process_mode_data(file_name: str) -> pd.DataFrame:
    """
    Import and clean the raw mode data from the csv file and return it as a dataframe.
    Translates column and index names to English and converts the dates to datetime format.
    Calculates daily distance and daily travel time per person for each mode.
    """

    # import the csv as a dataframe
    file = os.path.join("..", "data", "raw_data", file_name)
    df = pd.read_csv(file, delimiter=';', encoding='utf-8', skiprows=5, header=[0, 1, 2], index_col=[0, 1])
    df = df.iloc[:-1, :]                            # delete last row
    df.columns.names = [None] * df.columns.nlevels
    df.index.names = ['Date', 'Travel purpose']     # rename index

    # convert dates to datetime and fix index
    years = pd.to_datetime(df.index.get_level_values('Date').str.replace('*', '', regex=False).astype(int), format='%Y')
    df.index = pd.MultiIndex.from_arrays([years, df.index.get_level_values('Travel purpose')], names=['Date', 'Travel purpose'])

    # convert data to floats
    df = df.map(lambda x: str(x).replace(',', '.').strip() if pd.notna(x) else x)
    df = df.apply(pd.to_numeric, errors='coerce')

    # translate travel purposes in index
    purpose_mapping = {
        'Totaal': 'Total',
        'Van en naar het werk': 'Commuting',
        'Winkelen, boodschappen doen': 'Shopping',
        'Onderwijs volgen, cursus, kinderopvang': 'Education',
        'Vrije tijd': 'Leisure',
        'Overige reismotieven': 'Other'
    }
    cleaned_purposes = df.index.get_level_values('Travel purpose').map(lambda p: purpose_mapping.get(p, p))
    df.index = pd.MultiIndex.from_arrays([df.index.get_level_values('Date'), cleaned_purposes], names=['Date', 'Travel purpose'])

    # dictionaries for translating modes, indicators and units to English
    mode_mapping = {
        'Totaal': 'Total',
        'Auto (bestuurder)': 'Car (driver)',
        'Auto (passagier)': 'Car (passenger)',
        'Trein': 'Train',
        'Bus/tram/metro': 'Bus/tram/metro',
        'Fiets': 'Bicycle',
        'Lopen': 'Walking',
        'Overige vervoerwijze': 'Other modes'
    }

    indicator_mapping = {
        'Ritten per persoon per dag (gemiddeld)': 'Trips per person per day',
        'Afgelegde afstand per rit (gemiddeld)': 'Distance per trip',
        'Reisduur per rit': 'Travel time per trip'
    }

    unit_mapping = {
        'aantal': 'trips',
        'km': 'km',
        'Minuten': 'min'
    }

    # flatten from 3 header layers to 2 layers (incorporating unit in parentheses)
    new_columns = []
    for mode, indicator, unit in df.columns:
        trans_mode = mode_mapping.get(mode, mode)
        trans_indicator = indicator_mapping.get(indicator, indicator)
        trans_unit = unit_mapping.get(unit, unit)
        new_columns.append((trans_mode, f"{trans_indicator} ({trans_unit})"))

    df.columns = pd.MultiIndex.from_tuples(new_columns, names=['Mode', 'Indicator'])

    # add 'Car (total)' mode
    driver = 'Car (driver)'
    passenger = 'Car (passenger)'
    total_car = 'Car (total)'

    if driver in df.columns.get_level_values('Mode') and passenger in df.columns.get_level_values('Mode'):
        # Daily trips for driver and passenger
        trips_d = df[(driver, 'Trips per person per day (trips)')]
        trips_p = df[(passenger, 'Trips per person per day (trips)')]
        trips_tot = trips_d + trips_p
        df[(total_car, 'Trips per person per day (trips)')] = trips_tot

        # Weighted averages for distance and time per trip
        dist_d = df[(driver, 'Distance per trip (km)')]
        dist_p = df[(passenger, 'Distance per trip (km)')]
        df[(total_car, 'Distance per trip (km)')] = ((trips_d * dist_d + trips_p * dist_p) / trips_tot).fillna(0)

        time_d = df[(driver, 'Travel time per trip (min)')]
        time_p = df[(passenger, 'Travel time per trip (min)')]
        df[(total_car, 'Travel time per trip (min)')] = ((trips_d * time_d + trips_p * time_p) / trips_tot).fillna(0)

    # calculate daily distance and daily travel time per person for each mode
    all_modes = list(dict.fromkeys(df.columns.get_level_values('Mode')))
    for m in all_modes:
        trips_col = (m, 'Trips per person per day (trips)')
        dist_col = (m, 'Distance per trip (km)')
        time_col = (m, 'Travel time per trip (min)')

        if trips_col in df.columns and dist_col in df.columns:
            df[(m, 'Distance per person per day (km)')] = df[trips_col] * df[dist_col]

        if trips_col in df.columns and time_col in df.columns:
            df[(m, 'Travel time per person per day (min)')] = df[trips_col] * df[time_col]

    # sort columns alphabetically by mode
    df = df.sort_index(axis=1)

    return df

def export_dataframe(df: pd.DataFrame, file_name: str, index: bool = False) -> None:
    """
    Export the dataframe to a csv file in the processed_data folder.
    """
    file_path = os.path.join("..", "data", "processed_data", file_name)
    df.to_csv(file_path, index=index, sep=';', encoding='utf-8')


def process_income_data(income_file: str, population_file: str, cpi_file: str) -> pd.DataFrame:
    """
    Import household disposable income (CBS 85881NED), population (CBS 85524NED)
    and the overall CPI (CBS 83131NED) and return real disposable income per capita per year.
    """
    raw = os.path.join("..", "data", "raw_data")

    def read_annual(file_name, measure_code, column_name):
        # keep one measure and annual rows only
        df = pd.read_csv(os.path.join(raw, file_name), delimiter=';', encoding='utf-8')
        df = df[(df['Measure'] == measure_code) & df['Perioden'].str.endswith('JJ00')]
        years = df['Perioden'].str[:4].astype(int)
        return pd.Series(df['Value'].astype(float).values, index=years, name=column_name)

    income = read_annual(income_file, 'M006314', 'Disposable income (mln euro)')  # households, gross
    population = read_annual(population_file, 'M000351_2', 'Population (x1000)')  # total population

    # overall CPI
    cpi = pd.read_csv(os.path.join(raw, cpi_file), delimiter=';', encoding='utf-8', skiprows=5, header=None).iloc[:-1, :2]
    cpi.columns = ['Year', 'CPI_All']
    cpi = cpi.set_index(cpi['Year'].astype(int))['CPI_All'].str.replace(',', '.').astype(float)

    df = pd.concat([income, population, cpi], axis=1, join='inner')

    # real disposable income per person, in 2015 euros
    df['Real income per capita (euro 2015)'] = (
        df['Disposable income (mln euro)'] * 1e6
        / (df['Population (x1000)'] * 1e3)
        / (df['CPI_All'] / 100)
    )

    df.index = pd.to_datetime(df.index.astype(str), format='%Y')
    df.index.name = 'Date'
    return df

def process_fuel_weights(file_name: str, exclude_years: tuple[int, ...] = (2020,)) -> pd.DataFrame:
    """
    Import the raw car mileage data (CBS 80428NED) and return, per year, the share of
    car kilometres driven on petrol, diesel and LPG. These shares are used as weights
    to combine the three fuel CPIs into one fuel price index.
    Years in exclude_years (default: the COVID year 2020) are left out, so that later
    years are filled with the last unaffected year instead.
    """
    file_path = os.path.join("..", "data", "raw_data", file_name)
    df = pd.read_csv(file_path, delimiter=';', encoding='utf-8-sig', skiprows=4, decimal=',')
    df = df.iloc[:-1, :]                                    # delete last row
    df.columns = ['Ownership', 'Fuel', 'Year', 'Mln km']

    # remove asterisk of provisional figures and convert to numbers
    df['Year'] = df['Year'].str.replace('*', '', regex=False).astype(int)
    df['Mln km'] = pd.to_numeric(df['Mln km'], errors='coerce')

    # leave out years that are not representative (COVID)
    df = df[~df['Year'].isin(exclude_years)]

    # keep the three fuel types and translate them
    fuel_mapping = {'Benzine/overige': 'Petrol', 'Diesel': 'Diesel', 'LPG': 'LPG'}
    df = df[df['Fuel'].isin(fuel_mapping)]
    km = df.pivot(index='Year', columns='Fuel', values='Mln km').rename(columns=fuel_mapping)

    # share of total kilometres per fuel type
    weights = km.div(km.sum(axis=1), axis=0)
    weights.columns = [f'Weight_{c}' for c in weights.columns]

    weights.index = pd.to_datetime(weights.index.astype(str), format='%Y')
    weights.index.name = 'Date'
    return weights

def add_fuel_cpi(cpi_df: pd.DataFrame, weights_df: pd.DataFrame) -> pd.DataFrame:
    """Adds the weighted fuel CPI and the price ratio to cpi_df."""
    cpi = cpi_df.copy()

    # ensure date index is formatted as datetime
    if "Date" in cpi.columns:
        cpi["Date"] = pd.to_datetime(cpi["Date"])
        cpi = cpi.set_index("Date")
    cpi.index = pd.to_datetime(cpi.index)

    # align fuel weights to yearly frequency and fill any missing years
    fuels = ["Diesel", "Petrol", "LPG"]
    w = weights_df.copy()
    w.index = pd.to_datetime(w.index).year
    w = w.reindex(range(cpi.index.year.min(), cpi.index.year.max() + 1)).ffill().bfill()

    w_month = w.loc[cpi.index.year]
    w_month.index = cpi.index

    # calculate combined fuel price index and train-to-fuel price ratio
    cpi["CPI_Fuel"] = sum(cpi[f"CPI_{f}"] * w_month[f"Weight_{f}"] for f in fuels)
    cpi["Ratio"] = cpi["CPI_Train"] / cpi["CPI_Fuel"]
    return cpi.round(3)


def create_project_dataset(cpi_df: pd.DataFrame, mode_df: pd.DataFrame, income_df: pd.DataFrame, covid: tuple[int, int] = (2020, 2021), covid_ext_end: int = 2022,
) -> pd.DataFrame:
    """Creates the combined master table."""
    cpi_year = cpi_df.groupby(cpi_df.index.year)[["CPI_Train", "CPI_Fuel", "Ratio"]].mean()
    cpi_year.index.name = "year"

    total_mode = mode_df.xs("Total", level="Travel purpose")
    total_mode.index = pd.to_datetime(total_mode.index).year
    total_mode.index.name = "year"

    # convert modal distance and trip counts to floats
    train_km = pd.to_numeric(total_mode[("Train", "Distance per person per day (km)")], errors="coerce",).astype(float)
    car_km = pd.to_numeric(total_mode[("Car (total)", "Distance per person per day (km)")], errors="coerce",).astype(float)
    train_trips = pd.to_numeric(total_mode[("Train", "Trips per person per day (trips)")], errors="coerce",).astype(float)
    car_trips = pd.to_numeric(total_mode[("Car (total)", "Trips per person per day (trips)")], errors="coerce",).astype(float)

    df = pd.DataFrame({
        "train_km": train_km,
        "car_km": car_km,
        "train_trips": train_trips,
        "car_trips": car_trips,
    })
    df["train_share"] = (df["train_km"] / (df["train_km"] + df["car_km"])) * 100

    # clean and align annual real income per capita
    inc = income_df.copy()
    inc.index = pd.to_datetime(inc.index).year
    inc.index.name = "year"
    inc_col = inc[["Real income per capita (euro 2015)"]].rename(columns={"Real income per capita (euro 2015)": "real_income"})
    inc_col["real_income"] = pd.to_numeric(inc_col["real_income"], errors="coerce").astype(float)

    # merge mobility, prices, and income into a single annual dataset
    master = df.join(cpi_year, how="inner").join(inc_col, how="left")
    master["car_share"] = 100 - master["train_share"]

    # add dummy indicator variables for standard and extended COVID-19 periods
    master["covid"] = master.index.isin(range(covid[0], covid[1] + 1)).astype(int)
    master["covid_ext"] = master.index.isin(range(covid[0], covid_ext_end + 1)).astype(int)
    return master.round(3)

def process_mode_short(mode_df: pd.DataFrame) -> pd.DataFrame:
    """
    Short column names (mode, indicator) and an integer year + purpose index,
    so that the plots notebook can select data easily.
    """

    # mapping for column labels
    short = {
        "Distance per person per day (km)": "km_day",
        "Distance per trip (km)": "km_trip",
        "Travel time per person per day (min)": "min_day",
        "Travel time per trip (min)": "min_trip",
        "Trips per person per day (trips)": "trips_day",
    }
    mode = mode_df.copy()
    mode.columns = pd.MultiIndex.from_tuples([(m, short[i]) for m, i in mode_df.columns], names=["mode", "indicator"])
    mode.index = pd.MultiIndex.from_arrays([pd.to_datetime(mode_df.index.get_level_values(0)).year, mode_df.index.get_level_values(1)], names=["year", "purpose"],)
    
    return mode

In [7]:
# specify the file names and execute the functions

cpi_data_file_names = {
    "Train": "83131NED_TabelIndeling_20261002110845 - CPI Train.csv",
    "Diesel": "83131NED_TabelIndeling_20261002111330 - CPI Diesel.csv",
    "Petrol": "83131NED_TabelIndeling_20261002111353 - CPI Petrol.csv",
    "LPG": "83131NED_TabelIndeling_20261002111410 - CPI LPG.csv",
}
mode_data_file_name = "84755NED_TabelIndeling_20261007110933 - mode data.csv"
income_file = "85881NED_Observations - Income.csv"
population_file = "85524NED_Observations - Population.csv"
cpi_all_file = "83131NED_TabelIndeling_20261104142008 - CPI All.csv"
fuel_weights_file = "80428ned_TabelIndeling_20261104171023.csv"

weights_df = process_fuel_weights(fuel_weights_file)
cpi_raw_df = process_cpi_data(cpi_data_file_names)
cpi_df = add_fuel_cpi(cpi_raw_df, weights_df)
mode_df = process_mode_data(mode_data_file_name)
income_df = process_income_data(income_file, population_file, cpi_all_file)
project_df = create_project_dataset(cpi_df, mode_df, income_df)
cpi_year_df = process_cpi_yearly(cpi_df)
mode_short_df = process_mode_short(mode_df)

export_dataframe(cpi_df, "cpi_data.csv", index=True)
export_dataframe(mode_df, "mode_data.csv", index=True)
export_dataframe(income_df, "income_data.csv", index=True)
export_dataframe(weights_df, "fuel_weights.csv", index=True)
export_dataframe(project_df, "project_dataset.csv", index=True)
export_dataframe(cpi_year_df, "cpi_yearly.csv", index=True)
export_dataframe(mode_short_df, "mode_short.csv", index=True)